# Week 3 v2: Classical GNN + Quantum VQC Double Baseline

**Architecture upgrade:**
- ResNet-50 (ImageNet) → UNI (H&E pathology)
- GCNConv → GAT-Mamba
- VQC: 3-qubit, 2-layer, amplitude encoding + RY/RZ rotations

**Key fix:** MIL bags guaranteed balanced (no more all-negative test sets)

**Goal:** Establish v2 baselines before moving to Week 4-6 experiments.


## Cell 1 — Setup

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import roc_auc_score, f1_score
import time, json, sys, os

# VRAM management for RTX 5060 8GB
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache()

sys.path.insert(0, '..')

from torch_geometric.loader import DataLoader as PyGLoader
from torch_geometric.data import Data, Batch
from datasets import load_dataset
from scipy.spatial import KDTree

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT     = Path('..')
CKPT_DIR = ROOT / 'checkpoints'; CKPT_DIR.mkdir(exist_ok=True)
OUT_DIR  = ROOT / 'outputs';     OUT_DIR.mkdir(exist_ok=True)

torch.manual_seed(42)
np.random.seed(42)

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA: {torch.version.cuda}')

/home/kabi/.conda/envs/pathq/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
CUDA: 12.8


## Cell 2 — Load models

In [2]:
from pathq.model_v2 import QuantaPathV2
from pathq.uni_extractor import build_uni_extractor

print('Loading UNI...')
uni_model, uni_transform = build_uni_extractor(DEVICE)

# Note: Models will be instantiated for each experiment
print('\n✅ Models and extractors loaded')

[model_v2] Using Transformer for global branch
Loading UNI...
Loading UNI feature extractor from HuggingFace...
(First run downloads ~1.8GB — subsequent runs use cache)
UNI loaded: 303,350,784 params (all frozen)
Output dimension: 1024

✅ Models and extractors loaded


## Cell 3 — Build MIL bags with FIXED label assignment

In [3]:
# Cell 3 — Load pre-extracted UNI features

from pathq.dataset_v2 import get_loaders_from_features

FEAT_DIR = Path('./data/features_uni')  # ← Changed from ../data to ./data

print(f'Loading pre-extracted UNI features...')
print(f'Features dir: {FEAT_DIR}')
print(f'Features dir absolute: {FEAT_DIR.absolute()}')
print(f'Total feature files: {len(list(FEAT_DIR.glob("*_uni_features.pt")))}')
print()

train_loader, val_loader, test_loader = get_loaders_from_features(
    features_dir = FEAT_DIR,
    batch_size   = 4,
    k            = 8,
    seed         = 42,
    num_workers  = 0,
)

print(f'✅ Data loaders ready')
print(f'  Train: {len(train_loader)} batches')
print(f'  Val: {len(val_loader)} batches')
print(f'  Test: {len(test_loader)} batches')

Loading pre-extracted UNI features...
Features dir: data/features_uni
Features dir absolute: /home/kabi/PATHQ--Quantum-Digital-Pathology-for-Whole-Slide-Image-Analysis/notebooks/data/features_uni
Total feature files: 333

Split: train=233 (pos=78) val=50 (pos=17) test=50 (pos=16)
✅ Data loaders ready
  Train: 59 batches
  Val: 13 batches
  Test: 13 batches


## Cell 4 — Data loaders

## Cell 5 — Training functions

In [4]:
def train_one(model, loader, opt, device):
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        batch = batch.to(device)
        opt.zero_grad()
        logits, _ = model(batch)
        loss = F.cross_entropy(logits, batch.y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item()
        n += 1
    return total / max(n, 1)


@torch.no_grad()
def eval_model(model, loader, device):
    model.eval()
    all_probs, all_labels, total_loss, n = [], [], 0.0, 0
    for batch in loader:
        batch = batch.to(device)
        logits, _ = model(batch)
        total_loss += F.cross_entropy(logits, batch.y.view(-1)).item()
        all_probs.extend(torch.softmax(logits, 1)[:, 1].cpu().tolist())
        all_labels.extend(batch.y.view(-1).cpu().tolist())
        n += 1
    p, l  = np.array(all_probs), np.array(all_labels)
    preds = (p >= 0.5).astype(int)
    auc   = roc_auc_score(l, p) if len(np.unique(l)) > 1 else 0.5
    f1    = f1_score(l, preds, zero_division=0)
    tp = int(((preds==1)&(l==1)).sum())
    fn = int(((preds==0)&(l==1)).sum())
    tn = int(((preds==0)&(l==0)).sum())
    fp = int(((preds==1)&(l==0)).sum())
    return {
        'loss': total_loss / max(n, 1), 'auc': auc, 'f1': f1,
        'sensitivity': tp / max(tp+fn, 1),
        'specificity': tn / max(tn+fp, 1),
        'probs': p, 'labels': l
    }


def run_experiment(model, tr, va, te, device, label='run', epochs=30, lr=1e-4, ckpt=None):
    opt   = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=epochs, eta_min=1e-6
    )
    best_auc, no_imp = 0.0, 0

    print(f'\n{"Ep":>4} {"Loss":>8} {"AUC":>8} {"F1":>7} '
          f'{"Sens":>7} {"Spec":>7} {"Time":>7}  Best')
    print('─' * 65)

    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl = train_one(model, tr, opt, device)
        vm = eval_model(model, va, device)
        sched.step()
        elapsed  = time.time() - t0
        improved = vm['auc'] > best_auc

        print(f'{ep:4d} {tl:8.4f} {vm["auc"]:8.4f} {vm["f1"]:7.4f} '
              f'{vm["sensitivity"]:7.4f} {vm["specificity"]:7.4f} '
              f'{elapsed:6.1f}s  {"✓" if improved else ""}')

        if improved:
            best_auc = vm['auc']
            no_imp = 0
            if ckpt:
                torch.save({
                    'epoch': ep,
                    'model_state': model.state_dict(),
                    'val_auc': best_auc
                }, ckpt)
        else:
            no_imp += 1
            if no_imp >= 10:
                print(f'\nEarly stop at epoch {ep}')
                break

    print('─' * 65)
    print(f'Best val AUC: {best_auc:.4f}')

    if ckpt and Path(ckpt).exists():
        ckpt_data = torch.load(ckpt, weights_only=False)
        model.load_state_dict(ckpt_data['model_state'])

    test_metrics = eval_model(model, te, device)
    print(f'Test AUC    : {test_metrics["auc"]:.4f}  f1={test_metrics["f1"]:.4f}  '
          f'sens={test_metrics["sensitivity"]:.3f}  spec={test_metrics["specificity"]:.3f}')
    return test_metrics


print('✅ Training functions ready')

✅ Training functions ready


## Cell 6 — Experiment 1:  GAT+ TransMIL Baseline

In [ ]:
print('='*70)
print('Experiment 1: Classical GAT-Transformer Baseline (no VQC)')
print('='*70)
print('Config: 1040-dim inputs (UNI 1024 + pos.enc 16)')
print('        GAT-Transformer architecture')
print('        BAG_SIZE=16, batch_size=4')
print()

model_c = QuantaPathV2(use_vqc=False).to(DEVICE)
res_c = run_experiment(
    model_c, train_loader, val_loader, test_loader, DEVICE,
    label='classical_v2', epochs=30,
    ckpt=str(CKPT_DIR / 'v2_classical_best.pth')
)

print('\n✅ GAT+TransMIL(Transformers) complete')

Experiment 1: Classical GAT-Transformer Baseline (no VQC)
Config: 1040-dim inputs (UNI 1024 + pos.enc 16)
        GAT-Transformer architecture
        BAG_SIZE=16, batch_size=4

QuantaPathV2: use_vqc=False, trainable=1,227,906

  Ep     Loss      AUC      F1    Sens    Spec    Time  Best
─────────────────────────────────────────────────────────────────


/home/kabi/PATHQ--Quantum-Digital-Pathology-for-Whole-Slide-Image-Analysis/notebooks/../pathq/model_v2.py:86: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.global_enc = nn.TransformerEncoder(transformer_layer, num_layers=1)


   1   0.6102   0.7522  0.4348  0.2941  0.9697  131.8s  ✓
   2   0.5107   0.7629  0.4800  0.3529  0.9394  131.7s  ✓
   3   0.4501   0.8681  0.6897  0.5882  0.9394  131.7s  ✓
   4   0.5109   0.8253  0.4800  0.3529  0.9394  131.8s  
   5   0.4500   0.9216  0.7586  0.6471  0.9697  131.5s  ✓
   6   0.3718   0.9643  0.8485  0.8235  0.9394  130.6s  ✓
   7   0.3961   0.9643  0.8824  0.8824  0.9394  131.0s  
   8   0.2950   0.9340  0.7742  0.7059  0.9394  131.9s  
   9   0.3494   0.9679  0.8750  0.8235  0.9697  131.7s  ✓
  10   0.3727   0.9519  0.8235  0.8235  0.9091  131.6s  
  11   0.2107   0.9216  0.8276  0.7059  1.0000  131.6s  
  12   0.2496   0.8806  0.8000  0.7059  0.9697  131.6s  
  13   0.1919   0.9608  0.8333  0.8824  0.8788  133.7s  
  14   0.2742   0.9305  0.7879  0.7647  0.9091  132.6s  
  15   0.1418   0.9180  0.7742  0.7059  0.9394  130.7s  
  16   0.1507   0.8859  0.7500  0.7059  0.9091  130.6s  
  17   0.1266   0.9394  0.8000  0.8235  0.8788  131.2s  
  18   0.1369   0.9002  0

## Cell 7 — Experiment 2: Quantum Hybrid Model

In [ ]:
print('='*70)
print('Experiment 2: Quantum VQC + GAT-Transformer')
print('='*70)
print('Config: VQC: 3-qubit, 2-layer, amplitude encoding')
print('        Input to VQC: UNI 1024-dim → proj to 8-dim')
print('        VQC output: 3-dim (3-qubit measurement)')
print('        Total hybrid: 11-dim (proj 8 + quantum 3) + pos.enc 16 = 27-dim')
print()

model_q = QuantaPathV2(use_vqc=True, n_qubits=3, vqc_layers=2).to(DEVICE)
res_q = run_experiment(
    model_q, train_loader, val_loader, test_loader, DEVICE,
    label='quantum_v2', epochs=30,
    ckpt=str(CKPT_DIR / 'v2_quantum_best.pth')
)

print('\n✅ Quantum model complete')

Experiment 2: Quantum VQC + GAT-Transformer
Config: VQC: 3-qubit, 2-layer, amplitude encoding
        Input to VQC: UNI 1024-dim → proj to 8-dim
        VQC output: 3-dim (3-qubit measurement)
        Total hybrid: 11-dim (proj 8 + quantum 3) + pos.enc 16 = 27-dim

[VQC] lightning.qubit (3q, 2L)
QuantaPathV2: use_vqc=True, trainable=976,790

  Ep     Loss      AUC      F1    Sens    Spec    Time  Best
─────────────────────────────────────────────────────────────────


## Cell 8 — Results comparison

In [ ]:
print('\n' + '='*70)
print('FINAL RESULTS COMPARISON')
print('='*70)
print(f'\nMetric              Classical    Quantum      Delta')
print('─'*70)

metrics = ['auc', 'f1', 'sensitivity', 'specificity']
for metric in metrics:
    c_val = res_c[metric]
    q_val = res_q[metric]
    delta = q_val - c_val
    print(f'{metric.capitalize():15s}  {c_val:8.4f}      {q_val:8.4f}      {delta:+7.4f}')

print('─'*70)
print(f'\nQuantum advantage (AUC): {res_q["auc"] - res_c["auc"]:+.4f}')

# Save results
results_data = {
    'classical_auc': float(res_c['auc']),
    'quantum_auc': float(res_q['auc']),
    'classical_f1': float(res_c['f1']),
    'quantum_f1': float(res_q['f1']),
    'delta_auc': float(res_q['auc'] - res_c['auc']),
}

with open(OUT_DIR / 'v2_results.json', 'w') as f:
    json.dump(results_data, f, indent=2)

print(f'\n✅ Results saved: {OUT_DIR}/v2_results.json')

## Cell 9 — Sanity checks

In [ ]:
print('\nSANITY CHECKS')
print('='*70)

# Check 1: Test AUC > 0.5 (not random/broken)
assert res_c['auc'] > 0.5, f"Classical AUC {res_c['auc']:.4f} ≤ 0.5 — BROKEN"
print(f'✅ Check 1: Classical AUC {res_c["auc"]:.4f} > 0.5')

assert res_q['auc'] > 0.5, f"Quantum AUC {res_q['auc']:.4f} ≤ 0.5 — BROKEN"
print(f'✅ Check 2: Quantum AUC {res_q["auc"]:.4f} > 0.5')

# Check 2: F1 not all zeros (predictions mix of classes)
assert res_c['f1'] > 0.0, f"Classical F1 = 0 — all predictions same class"
print(f'✅ Check 3: Classical F1 {res_c["f1"]:.4f} > 0 (mixed predictions)')

assert res_q['f1'] > 0.0, f"Quantum F1 = 0 — all predictions same class"
print(f'✅ Check 4: Quantum F1 {res_q["f1"]:.4f} > 0 (mixed predictions)')

# Check 3: Sensitivity and specificity reasonable
print(f'✅ Check 5: Classical sensitivity={res_c["sensitivity"]:.3f}, specificity={res_c["specificity"]:.3f}')
print(f'✅ Check 6: Quantum sensitivity={res_q["sensitivity"]:.3f}, specificity={res_q["specificity"]:.3f}')

print('\n✅ All sanity checks passed')
print('\n' + '='*70)
print('Week 3 v2 COMPLETE')
print('='*70)